# Build Target Variable

Before building a model we need to define what **career success** means.

This notebook computes several candidate targets and shows the top 10 players for each,
so we can judge intuitively which definition best captures what we want to predict.

### Candidates
| # | Target | Type |
|---|---|---|
| A | Career Points Per Game (PPG) | Regression |
| B | Career Games Played | Regression |
| C | Career Win-Share Proxy (PPG × seasons) | Regression |
| D | All-Star / Top-75 flag | Classification |
| E | Composite Score (PPG + RPG + APG, normalized) | Regression |

In [ ]:
import sqlite3
import pandas as pd

DB_PATH = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\nba.sqlite"
CAREER_STATS_PATH = r"E:\Github\PS-Acq-Claude-Code-Test\jeff-NBA-model\player_career_stats.csv"

conn = sqlite3.connect(DB_PATH)
career = pd.read_csv(CAREER_STATS_PATH)
print(f"Career stats loaded: {len(career)} players")
career.head(3)

## Step 1 — Compute Points from Play-by-Play

The career stats CSV only has FGM. We need to split into 2PT and 3PT, and also get free throws made (FTM), to compute true points.

- **3PM:** eventmsgtype = 1, description contains '3PT'
- **FTM:** eventmsgtype = 3, description does NOT contain 'MISS'
- **2PM:** FGM − 3PM
- **Points:** 2×2PM + 3×3PM + FTM

In [ ]:
# Valid players (drafted 2000+) — same filter as build_player_stats
conn.execute("DROP TABLE IF EXISTS temp_valid_players")
conn.execute("""
    CREATE TEMPORARY TABLE temp_valid_players AS
    SELECT DISTINCT person_id AS player_id
    FROM draft_history
    WHERE season >= 2000
""")
conn.commit()

In [ ]:
# 3-pointers made: eventmsgtype=1 and description contains '3PT'
three_pm = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        COUNT(*) AS three_pm
    FROM play_by_play p
    JOIN temp_valid_players v ON p.player1_id = v.player_id
    WHERE p.eventmsgtype = 1
      AND (
          p.homedescription LIKE '%3PT%'
          OR p.visitordescription LIKE '%3PT%'
      )
    GROUP BY p.player1_id
""", conn)

print(f"Players with 3PM data: {len(three_pm)}")
three_pm.head()

In [ ]:
# Free throws made: eventmsgtype=3, description does NOT contain 'MISS'
ftm = pd.read_sql_query("""
    SELECT
        p.player1_id AS player_id,
        COUNT(*) AS ftm
    FROM play_by_play p
    JOIN temp_valid_players v ON p.player1_id = v.player_id
    WHERE p.eventmsgtype = 3
      AND p.player1_id IS NOT NULL
      AND p.player1_id != 0
      AND (
          (p.homedescription IS NOT NULL AND p.homedescription NOT LIKE '%MISS%')
          OR (p.visitordescription IS NOT NULL AND p.visitordescription NOT LIKE '%MISS%')
      )
    GROUP BY p.player1_id
""", conn)

print(f"Players with FTM data: {len(ftm)}")
ftm.head()

In [ ]:
# Merge into career stats and compute points
# Cast player_id to int64 in SQL results to match career CSV
three_pm['player_id'] = three_pm['player_id'].astype('int64')
ftm['player_id'] = ftm['player_id'].astype('int64')

df = career.merge(three_pm, on='player_id', how='left')
df = df.merge(ftm, on='player_id', how='left')
df[['three_pm', 'ftm']] = df[['three_pm', 'ftm']].fillna(0).astype(int)

df['two_pm']     = df['fgm'] - df['three_pm']
df['total_pts']  = 2 * df['two_pm'] + 3 * df['three_pm'] + df['ftm']

print(f"Players: {len(df)}")
df[['player_name', 'games_played', 'fgm', 'three_pm', 'ftm', 'total_pts']].head(10)

## Step 2 — Compute All Target Candidates

We require a minimum of **100 games played** to filter out players with tiny sample sizes.

In [ ]:
MIN_GAMES = 100
df_q = df[df['games_played'] >= MIN_GAMES].copy()
print(f"Players with {MIN_GAMES}+ games: {len(df_q)}")

# --- Target A: Career PPG ---
df_q['target_ppg'] = (df_q['total_pts'] / df_q['games_played']).round(1)

# --- Target B: Career Games Played ---
df_q['target_games'] = df_q['games_played']

# --- Target C: Win-Share Proxy (PPG x seasons) ---
df_q['target_ws_proxy'] = (df_q['target_ppg'] * df_q['seasons_played']).round(1)

# --- Target D: All-Star / Top-75 flag (from common_player_info) ---
players_info = pd.read_sql_query("""
    SELECT person_id AS player_id, greatest_75_flag
    FROM common_player_info
    WHERE draft_year >= 2000
""", conn)
players_info['player_id'] = players_info['player_id'].astype('int64')
df_q = df_q.merge(players_info, on='player_id', how='left')
df_q['target_top75'] = (df_q['greatest_75_flag'] == 'Y').astype(int)

# --- Target E: Composite Score (PPG + RPG + APG, z-score normalized) ---
df_q['rpg'] = (df_q['rebounds'] / df_q['games_played']).round(2)
df_q['apg'] = (df_q['assists']  / df_q['games_played']).round(2)

for col in ['target_ppg', 'rpg', 'apg']:
    df_q[f'{col}_z'] = (df_q[col] - df_q[col].mean()) / df_q[col].std()

df_q['target_composite'] = (df_q['target_ppg_z'] + df_q['rpg_z'] + df_q['apg_z']).round(2)

print("Targets computed.")

## Step 3 — Top 10 for Each Target

The name recognition test: do the top 10 look right for what we're trying to predict?

In [ ]:
# Target A — Career PPG
print("=== TARGET A: Career Points Per Game ===")
df_q.nlargest(10, 'target_ppg')[['player_name', 'games_played', 'seasons_played', 'target_ppg']]

In [ ]:
# Target B — Career Games Played
print("=== TARGET B: Career Games Played ===")
df_q.nlargest(10, 'target_games')[['player_name', 'games_played', 'seasons_played', 'target_ppg']]

In [ ]:
# Target C — Win-Share Proxy
print("=== TARGET C: Win-Share Proxy (PPG x Seasons) ===")
df_q.nlargest(10, 'target_ws_proxy')[['player_name', 'games_played', 'seasons_played', 'target_ppg', 'target_ws_proxy']]

In [ ]:
# Target D — Top 75 flag
print("=== TARGET D: Greatest 75 Flag ===")
top75 = df_q[df_q['target_top75'] == 1][['player_name', 'games_played', 'seasons_played', 'target_ppg']]
print(f"Players flagged as Top-75: {len(top75)}")
top75.sort_values('target_ppg', ascending=False).head(10)

In [ ]:
# Target E — Composite Score
print("=== TARGET E: Composite Score (PPG + RPG + APG normalized) ===")
df_q.nlargest(10, 'target_composite')[['player_name', 'games_played', 'seasons_played', 'target_ppg', 'rpg', 'apg', 'target_composite']]

## Step 4 — Distribution of Each Target

Check for skew, outliers, or heavy class imbalance before committing to a target.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df_q['target_ppg'].plot(kind='hist', bins=30, ax=axes[0,0], title='A: Career PPG')
df_q['target_games'].plot(kind='hist', bins=30, ax=axes[0,1], title='B: Career Games Played')
df_q['target_ws_proxy'].plot(kind='hist', bins=30, ax=axes[1,0], title='C: Win-Share Proxy')
df_q['target_composite'].plot(kind='hist', bins=30, ax=axes[1,1], title='E: Composite Score')

plt.tight_layout()
plt.show()

print("\nTarget D — Top 75 class balance:")
print(df_q['target_top75'].value_counts())